# 第72章 在线零售用户消费与RFM

使用 UCI Online Retail 20 万行公开交易样本，完成用户消费口径清洗、经营 KPI、消费集中度与 RFM 客群运营分析。

## 项目背景

英国在线零售商希望理解用户消费规模、复购与流失风险。数据来自 UCI Machine Learning Repository 的 Online Retail（原始 541,909 行），课程使用固定随机种子抽取的 200,000 行公开样本，并保留退货、取消和客户缺失等真实问题。

## 学习目标

- 建立可复核的交易清洗口径
- 从交易明细构造经营 KPI
- 识别商品与国家贡献集中度
- 用 RFM 形成可行动的客户分层
- 把结果写成决策建议而非图表描述


## 数据字典

| 字段 | 含义 | 使用说明 |
| --- | --- | --- |
| InvoiceNo | 发票号 | 以 C 开头通常为取消单 |
| StockCode | 商品编码 | 商品主键 |
| Description | 商品描述 | 存在缺失 |
| Quantity | 数量 | 负数通常表示退货 |
| InvoiceDate | 交易时间 | 英国当地时间 |
| UnitPrice | 单价 | 英镑 |
| CustomerID | 客户编号 | 部分缺失 |
| Country | 客户国家 | 订单归属地 |

## 数据质量检查清单

- 发票行是否重复
- 取消单、负数量和非正单价占比
- CustomerID 与 Description 缺失率
- 交易日期范围与异常时间
- KPI 是否仅基于有效正向销售


## 项目任务

1. 读取数据并建立质量基线
2. 定义有效销售口径并构造收入
3. 计算月度与总体 KPI
4. 分析商品和国家贡献集中度
5. 构建 RFM 并划分运营客群
6. 输出行动建议与限制


## 项目交付物

- 一份可复现的分析 Notebook
- 清洗规则与关键指标表
- 至少一张支持结论的图表
- 结论、限制和下一步建议

## 阶段检查点

- [ ] 问题和数据字典完成
- [ ] 质量检查和清洗记录完成
- [ ] 核心指标或图表完成
- [ ] 结论与限制完成

## 最低完成标准

- 每个代码阶段都有可见输出，不能依赖未展示的隐藏状态。
- 所有关键清洗、筛选和评价口径都写在 Markdown 或注释中。
- 最终结论至少引用一个数值或图表证据，并说明适用范围。

## 提升任务

完成基础验收后，可以增加一个对照方案、一个分组切片或一个参数敏感性实验，比较结果是否稳定。


## 1. 加载与质量审计

先保留原始问题并量化，避免清洗后无法解释样本变化。


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

df = pd.read_csv(f"{base_url}/datasets/uci_online_retail_200k.csv", parse_dates=["InvoiceDate"])
audit = pd.Series({
    "行数": len(df), "重复行": df.duplicated().sum(),
    "客户缺失": df["CustomerID"].isna().sum(),
    "取消单": df["InvoiceNo"].astype(str).str.startswith("C").sum(),
    "数量非正": (df["Quantity"] <= 0).sum(), "单价非正": (df["UnitPrice"] <= 0).sum()
})
print(audit.to_string()); print("日期:", df.InvoiceDate.min(), "至", df.InvoiceDate.max()); print(df.head())


## 2. 清洗口径与经营KPI

有效销售排除重复、取消、退货、非正价格；客户分析再要求客户编号非空。


In [ ]:
sales = df.drop_duplicates().copy()
valid = (~sales["InvoiceNo"].astype(str).str.startswith("C")) & (sales["Quantity"] > 0) & (sales["UnitPrice"] > 0)
sales = sales.loc[valid].copy()
sales["revenue"] = sales["Quantity"] * sales["UnitPrice"]
kpi = pd.Series({"有效明细":len(sales), "收入(GBP)":sales.revenue.sum(), "发票数":sales.InvoiceNo.nunique(),
                 "有ID客户数":sales.CustomerID.nunique(), "客单价":sales.groupby("InvoiceNo").revenue.sum().mean()})
print(kpi.round(2).to_string())
print(f"有效明细保留率: {len(sales)/len(df):.1%}")


## 3. 月度趋势与集中度

趋势用于发现变化，Pareto 指标用于判断经营是否依赖少数商品或市场。


In [ ]:
sales["month"] = sales.InvoiceDate.dt.to_period("M").astype(str)
monthly = sales.groupby("month").agg(revenue=("revenue", "sum"), invoices=("InvoiceNo", "nunique"))
product = sales.groupby("StockCode").revenue.sum().sort_values(ascending=False)
country = sales.groupby("Country").revenue.sum().sort_values(ascending=False)
top20_n = max(1, int(np.ceil(len(product)*.2)))
print(monthly.round(0)); print(f"前20%商品收入贡献: {product.head(top20_n).sum()/product.sum():.1%}")
print("主要国家贡献:\n", (country.head(8)/country.sum()).map(lambda x:f"{x:.1%}"))
monthly.revenue.plot(figsize=(9,4), marker="o", title="月度有效销售收入（GBP）"); plt.ylabel("GBP"); plt.tight_layout(); plt.show()


## 4. RFM客户分层

Recency 以数据末日次日为观察点；Frequency 使用不同发票数，Monetary 使用有效销售收入。


In [ ]:
customer_sales = sales.dropna(subset=["CustomerID"])
snapshot = customer_sales.InvoiceDate.max().normalize() + pd.Timedelta(days=1)
rfm = customer_sales.groupby("CustomerID").agg(
    recency = ("InvoiceDate", lambda x:(snapshot-x.max().normalize()).days),
    frequency = ("InvoiceNo", "nunique"), monetary=("revenue", "sum"))
rfm["r_score"] = pd.qcut(rfm.recency.rank(method="first"), 4, labels=[4,3,2,1]).astype(int)
rfm["f_score"] = pd.qcut(rfm.frequency.rank(method="first"), 4, labels=[1,2,3,4]).astype(int)
rfm["m_score"] = pd.qcut(rfm.monetary.rank(method="first"), 4, labels=[1,2,3,4]).astype(int)
rfm["segment"] = np.select([
    (rfm.r_score>=3)&(rfm.f_score>=3), (rfm.r_score<=2)&(rfm.f_score>=3),
    (rfm.r_score>=3)&(rfm.f_score<=2)], ["高价值活跃", "高价值待唤回", "新近低频"], default="一般/沉睡")
segment = rfm.groupby("segment").agg(customers=("monetary", "size"), revenue=("monetary", "sum"), median_recency=("recency", "median"))
segment["revenue_share"] = segment.revenue/segment.revenue.sum()
print(segment.sort_values("revenue", ascending=False).round(2))


## 5. 决策摘要

把指标转换成具体动作，同时保留抽样、缺失客户与退货口径的限制。


In [ ]:
best = segment.revenue.idxmax(); risk_n = int((rfm.segment=="高价值待唤回").sum())
print("经营建议")
print(f"1. 收入最高客群为「{best}」，优先设计分层权益而非全量促销。")
print(f"2. 对 {risk_n} 位高价值待唤回客户做小规模召回测试，并设置增量评估。")
print("3. 对高贡献商品建立缺货与退货监控，避免集中度转化为供应风险。")
print("限制：课程数据是原始数据的固定20万行样本；CustomerID缺失交易不能进入RFM；本分析描述关联，不证明营销动作的因果效果。")


## 结论与表达

- 清洗口径直接决定收入和客群结论，必须与结果一同交付。
- RFM 是运营排序工具，不是客户终身价值或因果响应模型。
- 集中度既意味着重点，也意味着供应与市场风险。


## 项目验收清单

- 能解释有效销售口径
- 能复算 KPI 与 Pareto 贡献
- 能说明 RFM 三指标观察窗口
- 建议包含目标客群、动作、指标和限制

建议重新启动内核后从第一个代码单元格运行，确认项目不依赖隐藏状态。


## 本章小结

使用 UCI Online Retail 20 万行公开交易样本，完成用户消费口径清洗、经营 KPI、消费集中度与 RFM 客群运营分析。


### 你已经完成

- 建立可复核的交易清洗口径
- 从交易明细构造经营 KPI
- 识别商品与国家贡献集中度
- 用 RFM 形成可行动的客户分层
- 把结果写成决策建议而非图表描述


### 项目流程速查

| 阶段 | 交付内容 |
| --- | --- |
| 步骤 1 | 读取数据并建立质量基线 |
| 步骤 2 | 定义有效销售口径并构造收入 |
| 步骤 3 | 计算月度与总体 KPI |
| 步骤 4 | 分析商品和国家贡献集中度 |
| 步骤 5 | 构建 RFM 并划分运营客群 |
| 步骤 6 | 输出行动建议与限制 |


### 质量与结论提醒

- 发票行是否重复
- 取消单、负数量和非正单价占比
- CustomerID 与 Description 缺失率
- 清洗口径直接决定收入和客群结论，必须与结果一同交付。
- RFM 是运营排序工具，不是客户终身价值或因果响应模型。
- 集中度既意味着重点，也意味着供应与市场风险。


### 项目交付检查

- [ ] 能解释有效销售口径
- [ ] 能复算 KPI 与 Pareto 贡献
- [ ] 能说明 RFM 三指标观察窗口
- [ ] 建议包含目标客群、动作、指标和限制


### 后续迭代建议

完成验收后，记录一个最值得继续验证的假设：可以是更多数据、不同时间窗口、另一种模型，或一个更细的分组分析。
